##### date_add()
- is used for **adding** a specified **number of days** to a **date** column in a DataFrame.

**Syntax**

     date_add(start_date, days)

- Adds a specified **number of days** to a **date** column.
  - `Positive value` → `Future date`
  - `Negative value` → `Past date`

**Arguments:** It takes two arguments
|   Argument   |     Description                                            |
|--------------|------------------------------------------------------------|  
| **start:**   | A `date column or a string` literal representing a `date`. |
| **days:**    | An `integer value or an integer column` representing the `number of days to add`. |
| **Return Value:** | It returns a `new column of DateType` with the calculated dates. |
| **Subtracting days:** | Passing a `negative integer` for the `days` argument will `subtract days from the date`. |

##### Difference: date_add() vs add_months()

| Function       | Adds   |
| -------------- | ------ |
| `date_add()`   | Days   |
| `add_months()` | Months |

In [0]:
from pyspark.sql.functions import col, current_date, date_add, datediff

##### 1) Add 10 Days to Current Date
- `Adds 10 days to today`.

In [0]:
spark.range(1).display()
spark.range(5).display()

id
0


id
0
1
2
3
4


In [0]:
df_add = spark.range(1) \
    .select(current_date().alias("today"),
            date_add(current_date(), 10).alias("after_10_days")
            )

display(df_add)

today,after_10_days
2026-03-07,2026-03-17


##### 2) Subtract 7 Days from Current Date
- `Negative value subtracts days`.

In [0]:
df_sub = spark.range(1) \
    .select(current_date().alias("today"),
            date_add(current_date(), -7).alias("before_7_days")
            )

display(df_sub)

today,before_7_days
2026-03-07,2026-02-28


##### 3) Subtract 30 Days from Column

In [0]:
data = [("2025-02-01",),
        ("2025-03-15",),
        ("2026-03-12",),
        ("2025-07-25",),
        ("2024-09-10",),
        ("2025-09-25",),
        ("2023-11-17",)]

df_sub_samp = spark.createDataFrame(data, ["order_date"]) \
    .withColumn("order_date", col("order_date").cast("date"))
    
display(df_sub_samp)

order_date
2025-02-01
2025-03-15
2026-03-12
2025-07-25
2024-09-10
2025-09-25
2023-11-17


In [0]:
df_sub_01 = df_sub_samp \
    .withColumn("previous_30_days", date_add("order_date", -30)) \
    .withColumn('diff_days', datediff(col('order_date'), col('previous_30_days')))

display(df_sub_01)

order_date,previous_30_days,diff_days
2025-02-01,2025-01-02,30
2025-03-15,2025-02-13,30
2026-03-12,2026-02-10,30
2025-07-25,2025-06-25,30
2024-09-10,2024-08-11,30
2025-09-25,2025-08-26,30
2023-11-17,2023-10-18,30


**Add 5 Days**

In [0]:
df_add_5d = df_sub_samp \
    .withColumn("delivery_date", date_add("order_date", 5)) \
    .withColumn('diff_days', datediff(col('delivery_date'), col('order_date')))

display(df_add_5d)

order_date,delivery_date,diff_days
2025-02-01,2025-02-06,5
2025-03-15,2025-03-20,5
2026-03-12,2026-03-17,5
2025-07-25,2025-07-30,5
2024-09-10,2024-09-15,5
2025-09-25,2025-09-30,5
2023-11-17,2023-11-22,5


##### 4) Dynamic Days (Using Another Column)
- `Days added based on another column value`.

In [0]:
data = [
    ("2025-02-01", 3),
    ("2025-03-10", 7),
    ("2025-05-20", 10),
    ("2025-07-25", 6),
    ("2025-08-30", 15),
    ("2025-10-10", 20),
    ("2025-12-22", 5),
]

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

schema = StructType([
    StructField("start_date", StringType(), True),
    StructField("days_to_add", IntegerType(), True)
])

df_dynamic = spark.createDataFrame(data, schema) \
    .withColumn("start_date", col("start_date").cast("date"))

df_dynamic_final = df_dynamic.withColumn("new_date", date_add("start_date", col("days_to_add")))
display(df_dynamic_final)

start_date,days_to_add,new_date
2025-02-01,3,2025-02-04
2025-03-10,7,2025-03-17
2025-05-20,10,2025-05-30
2025-07-25,6,2025-07-31
2025-08-30,15,2025-09-14
2025-10-10,20,2025-10-30
2025-12-22,5,2025-12-27


##### 5) Using SQL Query

In [0]:
# df_dynamic.createOrReplaceTempView("orders")

In [0]:
%sql
-- SELECT 
--     start_date,
--     date_add(start_date, 15) AS after_15_days
-- FROM orders

In [0]:
df_dynamic.createOrReplaceTempView("orders")

spark.sql("""
SELECT 
    start_date,
    last_day(start_date) AS last_day_of_month,
    date_add(start_date, 15) AS after_15_days
FROM orders
""").display()

start_date,last_day_of_month,after_15_days
2025-02-01,2025-02-28,2025-02-16
2025-03-10,2025-03-31,2025-03-25
2025-05-20,2025-05-31,2025-06-04
2025-07-25,2025-07-31,2025-08-09
2025-08-30,2025-08-31,2025-09-14
2025-10-10,2025-10-31,2025-10-25
2025-12-22,2025-12-31,2026-01-06


##### 6) Date Arithmetic (Adding & Subtracting Dates)

In [0]:
from pyspark.sql.functions import date_add, date_sub, add_months, last_day

In [0]:
spark.range(1).select(
    current_date().alias("today"),
    date_add(current_date(), 10).alias("plus_10_days"),
    date_sub(current_date(), 5).alias("minus_5_days"),
    last_day(current_date()).alias("last_day_of_month")
).display()

today,plus_10_days,minus_5_days,last_day_of_month
2026-03-07,2026-03-17,2026-03-02,2026-03-31


In [0]:
%sql
SELECT
    current_date() AS today,
    date_add(current_date(), 10) AS plus_10_days,
    date_sub(current_date(), 5) AS minus_5_days,
    add_months(current_date(), 2) AS plus_2_months,
    add_months(current_date(), -2) AS minus_2_months,
    last_day(current_date()) AS last_day_of_month;

today,plus_10_days,minus_5_days,plus_2_months,minus_2_months,last_day_of_month
2026-03-07,2026-03-17,2026-03-02,2026-05-07,2026-01-07,2026-03-31
